In [1]:
print("bonjour")

bonjour


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls "/content/drive/MyDrive"

'affectation groupes APP0 23_24.gdoc'
'affectation groupes APP0 23_24.pdf'
'Aigis 20k'
'A national rabies vaccination campaign in Tunisia....gsheet'
'Colab Notebooks'
'Copie de Fitness Gym Social Media Strategy by Slidesgo.gslides'
'demande_stage_231JFT6860_1781263032542-converted (2).gdoc'
'Diapositive 2 – What is the Problem?'$'\n''⚠️ Key Issue... (1).gsheet'
'Diapositive 2 – What is the Problem?'$'\n''⚠️ Key Issue....gsheet'
'Examen-1A-2020_2021_version A_2020_2021 (1).pdf'
'Google AI Studio'
'Liste des entreprises_PFE2023-2024.xlsx'
 MAKEUP_2022122819284293.jpg
 MonProjet
'Présentation sans titre.gslides'
 stagedeeplearning
'tictactoe rania.zip'


In [4]:
!ls "/content/drive/MyDrive"

'affectation groupes APP0 23_24.gdoc'
'affectation groupes APP0 23_24.pdf'
'Aigis 20k'
'A national rabies vaccination campaign in Tunisia....gsheet'
'Colab Notebooks'
'Copie de Fitness Gym Social Media Strategy by Slidesgo.gslides'
'demande_stage_231JFT6860_1781263032542-converted (2).gdoc'
'Diapositive 2 – What is the Problem?'$'\n''⚠️ Key Issue... (1).gsheet'
'Diapositive 2 – What is the Problem?'$'\n''⚠️ Key Issue....gsheet'
'Examen-1A-2020_2021_version A_2020_2021 (1).pdf'
'Google AI Studio'
'Liste des entreprises_PFE2023-2024.xlsx'
 MAKEUP_2022122819284293.jpg
 MonProjet
'Présentation sans titre.gslides'
 stagedeeplearning
'tictactoe rania.zip'


In [5]:
!ls "/content/drive/MyDrive"

'affectation groupes APP0 23_24.gdoc'
'affectation groupes APP0 23_24.pdf'
'Aigis 20k'
'A national rabies vaccination campaign in Tunisia....gsheet'
'Colab Notebooks'
'Copie de Fitness Gym Social Media Strategy by Slidesgo.gslides'
'demande_stage_231JFT6860_1781263032542-converted (2).gdoc'
'Diapositive 2 – What is the Problem?'$'\n''⚠️ Key Issue... (1).gsheet'
'Diapositive 2 – What is the Problem?'$'\n''⚠️ Key Issue....gsheet'
'Examen-1A-2020_2021_version A_2020_2021 (1).pdf'
'Google AI Studio'
'Liste des entreprises_PFE2023-2024.xlsx'
 MAKEUP_2022122819284293.jpg
 MonProjet
'Présentation sans titre.gslides'
 stagedeeplearning
'tictactoe rania.zip'


In [6]:
!ls "/content/drive/MyDrive"



'affectation groupes APP0 23_24.gdoc'
'affectation groupes APP0 23_24.pdf'
'Aigis 20k'
'A national rabies vaccination campaign in Tunisia....gsheet'
'Colab Notebooks'
'Copie de Fitness Gym Social Media Strategy by Slidesgo.gslides'
'demande_stage_231JFT6860_1781263032542-converted (2).gdoc'
'Diapositive 2 – What is the Problem?'$'\n''⚠️ Key Issue... (1).gsheet'
'Diapositive 2 – What is the Problem?'$'\n''⚠️ Key Issue....gsheet'
'Examen-1A-2020_2021_version A_2020_2021 (1).pdf'
'Google AI Studio'
'Liste des entreprises_PFE2023-2024.xlsx'
 MAKEUP_2022122819284293.jpg
 MonProjet
'Présentation sans titre.gslides'
 stagedeeplearning
'tictactoe rania.zip'


In [7]:
!ls "/content/drive/MyDrive/Aigis 20k/Model"

agis_model_folders.py  convert_model.py  New_Aigis.py
benchmark_results      Dataset.zip	 NewTestApp.py


In [8]:
!unzip -q "/content/drive/MyDrive/Aigis 20k/Model/Dataset.zip" -d "/content/dataset"

In [9]:
!ls "/content/dataset"

corrosion  cracks  normal  spalling  test  train  val


In [10]:
!ls "/content/dataset"

corrosion  cracks  normal  spalling  test  train  val


In [11]:
!ls "/content/dataset/train"

corrosion  cracks  normal  spalling


In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import json
import time
from datetime import datetime
import glob

# Seed pour reproductibilité
import random
random.seed(42); np.random.seed(42); tf.random.set_seed(42)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
benchmark_dir = f"/content/benchmark_results/run_{timestamp}"
os.makedirs(benchmark_dir, exist_ok=True)
models_dir = os.path.join(benchmark_dir, 'models')
plots_dir = os.path.join(benchmark_dir, 'plots')
metrics_dir = os.path.join(benchmark_dir, 'metrics')
os.makedirs(models_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

# Chemins adaptés à Colab (confirmés par tes !ls précédents)
dataset_path = "/content/dataset"
train_path = os.path.join(dataset_path, 'train')
val_path = os.path.join(dataset_path, 'val')
test_path = os.path.join(dataset_path, 'test')

class_names = ['corrosion', 'cracks', 'normal', 'spalling']
num_classes = len(class_names)

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=25, width_shift_range=0.15,
    height_shift_range=0.15, shear_range=0.15, zoom_range=0.3,
    horizontal_flip=True, fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_path, target_size=(224, 224), batch_size=32,
    class_mode='categorical', shuffle=True, seed=42
)
val_generator = val_datagen.flow_from_directory(
    val_path, target_size=(224, 224), batch_size=32,
    class_mode='categorical', shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    test_path, target_size=(224, 224), batch_size=32,
    class_mode='categorical', shuffle=False
)

class_weights = class_weight.compute_class_weight(
    class_weight='balanced', classes=np.unique(train_generator.classes), y=train_generator.classes
)
class_weights_dict = dict(zip(range(num_classes), class_weights))

def create_model(num_classes):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    for layer in base_model.layers[:143]:
        layer.trainable = True
    model = models.Sequential([
        base_model, layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'), layers.BatchNormalization(),
        layers.Dropout(0.7), layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = create_model(num_classes)
model.summary()

checkpoint = ModelCheckpoint(os.path.join(models_dir, 'best_model_folders.keras'), monitor='val_loss', save_best_only=True, mode='min')
early_stopping = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, verbose=1)

history = model.fit(
    train_generator, validation_data=val_generator, epochs=50,
    class_weight=class_weights_dict,
    callbacks=[checkpoint, early_stopping, lr_scheduler]
)

model.save(os.path.join(models_dir, 'agis_folders.keras'))
print(f"✅ Terminé ! Résultats dans : {benchmark_dir}")

Found 14707 images belonging to 4 classes.
Found 3152 images belonging to 4 classes.
Found 3154 images belonging to 4 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         2,052 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,640,900 (94.00 MB)

 Trainable params: 24,586,756 (93.79 MB)

 Non-trainable params: 54,144 (211.50 KB)

Epoch 1/50
225/460 ━━━━━━━━━━━━━━━━━━━━ 1:01:48 16s/step - accuracy: 0.6720 - loss: 1.1125

In [ ]:
!ls "/content/drive/MyDrive/Aigis 20k" 2>&1
!ls "/content/dataset" 2>&1



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls "/content/drive/MyDrive"


In [ ]:
!unzip -q "/content/drive/MyDrive/Aigis 20k/Model/Dataset.zip" -d "/content/dataset"

In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import json
import time
from datetime import datetime
import glob

# Seed pour reproductibilité
import random
random.seed(42); np.random.seed(42); tf.random.set_seed(42)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
benchmark_dir = f"/content/benchmark_results/run_{timestamp}"
os.makedirs(benchmark_dir, exist_ok=True)
models_dir = os.path.join(benchmark_dir, 'models')
plots_dir = os.path.join(benchmark_dir, 'plots')
metrics_dir = os.path.join(benchmark_dir, 'metrics')
os.makedirs(models_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

# Chemins adaptés à Colab (confirmés par tes !ls précédents)
dataset_path = "/content/dataset"
train_path = os.path.join(dataset_path, 'train')
val_path = os.path.join(dataset_path, 'val')
test_path = os.path.join(dataset_path, 'test')

class_names = ['corrosion', 'cracks', 'normal', 'spalling']
num_classes = len(class_names)

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=25, width_shift_range=0.15,
    height_shift_range=0.15, shear_range=0.15, zoom_range=0.3,
    horizontal_flip=True, fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_path, target_size=(224, 224), batch_size=32,
    class_mode='categorical', shuffle=True, seed=42
)
val_generator = val_datagen.flow_from_directory(
    val_path, target_size=(224, 224), batch_size=32,
    class_mode='categorical', shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    test_path, target_size=(224, 224), batch_size=32,
    class_mode='categorical', shuffle=False
)

class_weights = class_weight.compute_class_weight(
    class_weight='balanced', classes=np.unique(train_generator.classes), y=train_generator.classes
)
class_weights_dict = dict(zip(range(num_classes), class_weights))

def create_model(num_classes):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    for layer in base_model.layers[:143]:
        layer.trainable = True
    model = models.Sequential([
        base_model, layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'), layers.BatchNormalization(),
        layers.Dropout(0.7), layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = create_model(num_classes)
model.summary()

checkpoint = ModelCheckpoint(os.path.join(models_dir, 'best_model_folders.keras'), monitor='val_loss', save_best_only=True, mode='min')
early_stopping = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, verbose=1)

history = model.fit(
    train_generator, validation_data=val_generator, epochs=50,
    class_weight=class_weights_dict,
    callbacks=[checkpoint, early_stopping, lr_scheduler]
)

model.save(os.path.join(models_dir, 'agis_folders.keras'))
print(f"✅ Terminé ! Résultats dans : {benchmark_dir}")

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Évaluation sur le set de validation
val_loss, val_accuracy = model.evaluate(val_generator)
print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

# Évaluation sur le TEST set (le vrai chiffre qu'on cherche)
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

# Prédictions détaillées sur le test set
test_pred = model.predict(test_generator)
test_pred_classes = np.argmax(test_pred, axis=1)
test_true = test_generator.classes

print("\n=== Rapport de classification (Test set) ===")
print(classification_report(test_true, test_pred_classes, target_names=class_names))

# Matrice de confusion
test_cm = confusion_matrix(test_true, test_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Test Confusion Matrix - Ma reproduction')
plt.savefig(os.path.join(plots_dir, 'test_confusion_matrix.png'))
plt.show()

# Sauvegarde de la config pour comparaison
config = {
    'test_accuracy': float(test_accuracy),
    'test_loss': float(test_loss),
    'validation_accuracy': float(val_accuracy),
    'validation_loss': float(val_loss),
    'epochs_run': len(history.history['loss']),
}
with open(os.path.join(metrics_dir, 'my_run_config.json'), 'w') as f:
    json.dump(config, f, indent=4)

print(f"\n🎯 COMPARAISON : ton test_accuracy = {test_accuracy:.4f} vs original = 0.9566")

In [ ]:
# === ÉTAPE 1 : Remonter le Drive ===
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# === ÉTAPE 2 : Redézipper le dataset (5-15 min, patience) ===
!unzip -q "/content/drive/MyDrive/Aigis 20k/Model/Dataset.zip" -d "/content/dataset"
print("✅ Dézippage terminé")

In [ ]:
# === ÉTAPE 3 : Visualisations du dataset ===
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

dataset_path = "/content/dataset"
class_names = ['corrosion', 'cracks', 'normal', 'spalling']
splits = ['train', 'val', 'test']

# Comptage des images par classe et par split
counts = {split: {} for split in splits}
for split in splits:
    for cls in class_names:
        folder = os.path.join(dataset_path, split, cls)
        counts[split][cls] = len(os.listdir(folder)) if os.path.exists(folder) else 0

print("=== Vérification des comptages ===")
for split in splits:
    print(f"{split}: {counts[split]}")

# --- Graphique en barres : distribution par split ---
fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(class_names))
width = 0.25
colors = ['#4C72B0', '#DD8452', '#55A868']
for i, split in enumerate(splits):
    values = [counts[split][cls] for cls in class_names]
    ax.bar([p + width*i for p in x], values, width, label=split, color=colors[i])
ax.set_xticks([p + width for p in x])
ax.set_xticklabels(class_names)
ax.set_ylabel("Nombre d'images")
ax.set_title("Distribution des classes par split (train/val/test)")
ax.legend()
plt.tight_layout()
plt.savefig('/content/dataset_distribution.png', dpi=150)
plt.show()

# --- Exemples d'images par classe ---
fig, axes = plt.subplots(len(class_names), 5, figsize=(15, 12))
for row, cls in enumerate(class_names):
    folder = os.path.join(dataset_path, 'train', cls)
    images = os.listdir(folder)
    sample = random.sample(images, min(5, len(images)))
    for col, img_name in enumerate(sample):
        img = Image.open(os.path.join(folder, img_name))
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
    axes[row, 0].set_title(cls, fontsize=14, loc='left')
plt.suptitle("Exemples d'images par classe (train)", fontsize=16)
plt.tight_layout()
plt.savefig('/content/dataset_samples.png', dpi=150)
plt.show()

# --- Camembert : répartition globale ---
total_counts = {cls: sum(counts[split][cls] for split in splits) for cls in class_names}
plt.figure(figsize=(7, 7))
plt.pie(
    total_counts.values(),
    labels=total_counts.keys(),
    autopct='%1.1f%%',
    colors=['#4C72B0', '#DD8452', '#55A868', '#C44E52']
)
plt.title("Répartition globale des 4 classes")
plt.savefig('/content/dataset_pie.png', dpi=150)
plt.show()

In [ ]:
!python 01_eda_data_cleaning.py

In [ ]:
BACKBONE_NAME = "resnet50"

In [ ]:
!python 02_gridsearch_train_eval.py